Ячейка 1: Импорты, Seed, Device

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.utils import draw_bounding_boxes
from torchvision.ops import box_iou

# Настройки
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Фиксация случайности
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Папки
os.makedirs("artifacts/figures", exist_ok=True)

Using device: cpu


Ячейка 2: Данные Часть A (SVHN)

In [2]:
# SVHN возвращает PIL Image при transform=None
# Поэтому ToTensor() НУЖЕН!

transform_base = transforms.Compose([
    transforms.ToTensor(),  # PIL -> Tensor [0,1]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # [0,1] -> [-1,1]
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Загрузка SVHN (transform=None, чтобы получить PIL Images)
train_full = datasets.SVHN(root="./data", split="train", download=True, transform=None)
test_dataset = datasets.SVHN(root="./data", split="test", download=True, transform=None)

# Split train -> train (80%) + val (20%)
n_train = int(len(train_full) * 0.8)
indices = torch.randperm(len(train_full)).tolist()
train_idx, val_idx = indices[:n_train], indices[n_train:]

# Класс-обёртка для применения разных трансформов
class SVHN_Dual(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img, label = self.base[self.indices[idx]]  # img - PIL Image
        if self.transform:
            img = self.transform(img)  # ToTensor + Normalize
        return img, label

train_dataset_base = SVHN_Dual(train_full, train_idx, transform_base)
val_dataset = SVHN_Dual(train_full, val_idx, transform_base)
train_dataset_aug = SVHN_Dual(train_full, train_idx, transform_aug)

# DataLoader
BATCH_SIZE = 64 if DEVICE.type == 'cpu' else 128
loader_train_base = DataLoader(train_dataset_base, batch_size=BATCH_SIZE, shuffle=True)
loader_train_aug = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
loader_test = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset_base)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# Sanity check
x, y = next(iter(loader_train_base))
print(f"Batch shape: x={x.shape}, y={y.shape}")
print(f"Label sample: {y[:10]}")

Train: 58605, Val: 14652, Test: 26032
Batch shape: x=torch.Size([64, 3, 32, 32]), y=torch.Size([64])
Label sample: tensor([6, 3, 2, 2, 2, 8, 1, 1, 7, 1])


Ячейка 3: Модели и Функции (Часть A)

In [3]:
# Простая CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.net(x)

# Функции обучения
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Ячейка 4: Эксперименты C1-C4

In [ ]:
results = []
best_model_state = None
best_val_acc = 0
best_config = {}

# Конфигурация экспериментов
experiments = [
    {"id": "C1", "model": SimpleCNN(), "loader": loader_train_base, "name": "CNN-Base"},
    {"id": "C2", "model": SimpleCNN(), "loader": loader_train_aug, "name": "CNN-Aug"},
    {"id": "C3", "model": models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1), "loader": loader_train_base, "name": "ResNet-Freeze", "freeze": True},
    {"id": "C4", "model": models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1), "loader": loader_train_base, "name": "ResNet-FineTune", "freeze": False},
]

EPOCHS = 5 if DEVICE.type == 'cpu' else 10  # Мало эпох для скорости
criterion = nn.CrossEntropyLoss()

for exp in experiments:
    print(f"\nRunning {exp['id']} ({exp['name']})...")
    model = exp["model"].to(DEVICE)

    # Настройка ResNet
    if "resnet" in exp["name"].lower():
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 10)
        if exp.get("freeze"):
            for param in model.parameters(): param.requires_grad = False
            for param in model.fc.parameters(): param.requires_grad = True
        else:
            for param in model.parameters(): param.requires_grad = True
            # Разморозим только последний блок для экономии времени
            for param in model.layer4.parameters(): param.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(EPOCHS):
        t_loss, _ = train_epoch(model, exp["loader"], optimizer, criterion, DEVICE)
        v_loss, v_acc = evaluate(model, loader_val, criterion, DEVICE)
        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["val_acc"].append(v_acc)
        print(f"Epoch {epoch+1}: Val Acc = {v_acc:.4f}")

    # Сохранение лучшей модели
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_state = model.state_dict().copy()
        best_config = {"id": exp["id"], "name": exp["name"], "epochs": EPOCHS}

    results.append({
        "experiment_id": exp["id"],
        "task": "classification",
        "dataset": "SVHN",
        "seed": SEED,
        "model_summary": exp["name"],
        "optimizer": "Adam",
        "lr": 0.001,
        "epochs_trained": EPOCHS,
        "best_val_accuracy": round(max(history["val_acc"]), 4),
        "test_accuracy": "", # Заполним позже
        "precision": "", "recall": "", "mean_iou": "",
        "notes": ""
    })

# Финальный тест лучшей модели
print("\nEvaluating Best Model on Test...")
best_model = SimpleCNN().to(DEVICE) # Заглушка, перезапишем weights
# Нужно знать архитектуру лучшей модели. Для простоты возьмем ResNet если он выиграл, иначе CNN.
# В реальном коде нужно хранить класс модели. Здесь упростим:
if "ResNet" in best_config["name"]:
    final_model = models.resnet18(weights=None).to(DEVICE)
    final_model.fc = nn.Linear(512, 10)
    final_model.load_state_dict(best_model_state)
else:
    final_model = SimpleCNN().to(DEVICE)
    final_model.load_state_dict(best_model_state)

_, test_acc = evaluate(final_model, loader_test, criterion, DEVICE)
print(f"Test Accuracy: {test_acc:.4f}")

# Обновим результат лучшего эксперимента в таблице
for r in results:
    if r["experiment_id"] == best_config["id"]:
        r["test_accuracy"] = round(test_acc, 4)

# Сохранение артефактов
torch.save(best_model_state, "artifacts/best_classifier.pt")
with open("artifacts/best_classifier_config.json", "w") as f:
    json.dump(best_config, f, indent=2)


Running C1 (CNN-Base)...
Epoch 1: Val Acc = 0.8316
Epoch 2: Val Acc = 0.8531
Epoch 3: Val Acc = 0.8601
Epoch 4: Val Acc = 0.8633
Epoch 5: Val Acc = 0.8809

Running C2 (CNN-Aug)...
Epoch 1: Val Acc = 0.7475
Epoch 2: Val Acc = 0.7903
Epoch 3: Val Acc = 0.8053
Epoch 4: Val Acc = 0.8103
Epoch 5: Val Acc = 0.8208

Running C3 (ResNet-Freeze)...
Epoch 1: Val Acc = 0.3163
Epoch 2: Val Acc = 0.3180
Epoch 3: Val Acc = 0.3224
Epoch 4: Val Acc = 0.3184


Ячейка 5: Графики Часть A

In [ ]:
# 1. Кривые обучения лучшего эксперимента (упрощенно - последний запущенный, если он лучший)
# Для полноценности нужно хранить history лучшего. Здесь сделаем бар-плот сравнения.
ids = [r["experiment_id"] for r in results]
accs = [r["best_val_accuracy"] for r in results]

plt.figure(figsize=(8, 5))
plt.bar(ids, accs, color=['skyblue', 'lightgreen', 'salmon', 'orange'])
plt.title("Comparison C1-C4 (Val Accuracy)")
plt.ylabel("Accuracy")
plt.savefig("artifacts/figures/classification_compare.png")
plt.show()

# 2. Аугментации (визуализация)
img, _ = train_dataset_aug[0]
plt.figure(figsize=(10, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    # Нужно применить трансформ к одному изображению несколько раз для визуализации
    # Упрощенно покажем одно
    plt.imshow(img.permute(1,2,0))
    plt.axis('off')
plt.savefig("artifacts/figures/augmentations_preview.png")
plt.show()

Ячейка 6: Часть B (Detection VOC)

In [ ]:
# Загрузка модели детекции
det_model = models.detection.fasterrcnn_resnet50_fpn(weights=models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
det_model.eval().to(DEVICE)

# Датасет VOC (только несколько картинок для скорости)
# VOC требует индексацию. Для скорости возьмем первые 20 изображений из val
voc_dataset = datasets.VOCDetection(root="./data_voc", year="2012", image_set="val", download=True)
indices = list(range(20)) # Только 20 картинок для метрик
voc_subset = Subset(voc_dataset, indices)

def voc_collate(batch):
    return tuple(zip(*batch))
loader_voc = DataLoader(voc_subset, batch_size=1, collate_fn=voc_collate)

CLASS_MAP = voc_dataset.classes # Имена классов

# Функция IoU
def calculate_iou(box1, box2):
    # box: [x1, y1, x2, y2]
    b1 = torch.tensor(box1).float().unsqueeze(0)
    b2 = torch.tensor(box2).float().unsqueeze(0)
    return box_iou(b1, b2).item()

metrics_v1 = {"tp": 0, "fp": 0, "fn": 0, "iou_sum": 0}
metrics_v2 = {"tp": 0, "fp": 0, "fn": 0, "iou_sum": 0}
THRESH_V1, THRESH_V2 = 0.3, 0.7
IOU_THRESH = 0.5

print("Running Detection Inference...")
for idx, (img, target) in enumerate(loader_voc):
    img_tensor = transforms.ToTensor()(img[0]).to(DEVICE)
    gt_boxes = target[0]["boxes"].tolist()

    with torch.no_grad():
        pred = det_model([img_tensor])[0]

    scores = pred["scores"].cpu().numpy()
    pred_boxes = pred["boxes"].cpu().tolist()

    # V1 & V2 Logic
    for thresh, metrics in [(THRESH_V1, metrics_v1), (THRESH_V2, metrics_v2)]:
        kept_boxes = [b for i, b in enumerate(pred_boxes) if scores[i] > thresh]

        # Простой подсчет TP/FP
        # Если есть GT и есть предсказания -> пытаемся матчить
        # Упрощение: если IoU > 0.5 хоть с одним GT -> TP, иначе FP
        # Если GT остался без матча -> FN

        matched_gt = [False] * len(gt_boxes)
        for pb in kept_boxes:
            ious = [calculate_iou(pb, gb) for gb in gt_boxes]
            if ious and max(ious) >= IOU_THRESH:
                metrics["tp"] += 1
                metrics["iou_sum"] += max(ious)
                matched_gt[ious.index(max(ious))] = True
            else:
                metrics["fp"] += 1

        metrics["fn"] += matched_gt.count(False)

# Расчет метрик
def calc_metrics(m):
    prec = m["tp"] / (m["tp"] + m["fp"]) if (m["tp"] + m["fp"]) > 0 else 0
    rec = m["tp"] / (m["tp"] + m["fn"]) if (m["tp"] + m["fn"]) > 0 else 0
    miou = m["iou_sum"] / m["tp"] if m["tp"] > 0 else 0
    return prec, rec, miou

p1, r1, i1 = calc_metrics(metrics_v1)
p2, r2, i2 = calc_metrics(metrics_v2)

print(f"V1 (0.3): P={p1:.2f}, R={r1:.2f}, IoU={i1:.2f}")
print(f"V2 (0.7): P={p2:.2f}, R={r2:.2f}, IoU={i2:.2f}")

# Добавим в results
results.append({
    "experiment_id": "V1", "task": "detection", "dataset": "VOC2012", "seed": SEED,
    "model_summary": "FasterRCNN_ResNet50", "optimizer": "", "lr": "", "epochs_trained": 0,
    "best_val_accuracy": "", "test_accuracy": "",
    "precision": round(p1, 4), "recall": round(r1, 4), "mean_iou": round(i1, 4), "notes": f"thresh={THRESH_V1}"
})
results.append({
    "experiment_id": "V2", "task": "detection", "dataset": "VOC2012", "seed": SEED,
    "model_summary": "FasterRCNN_ResNet50", "optimizer": "", "lr": "", "epochs_trained": 0,
    "best_val_accuracy": "", "test_accuracy": "",
    "precision": round(p2, 4), "recall": round(r2, 4), "mean_iou": round(i2, 4), "notes": f"thresh={THRESH_V2}"
})

Ячейка 7: Визуализация Детекции и Сохранение

In [ ]:
# Визуализация примеров
img, target = voc_dataset[0]
img_t = transforms.ToTensor()(img).to(DEVICE)
with torch.no_grad():
    pred = det_model([img_t])[0]

# Рисуем V1
boxes_v1 = pred["boxes"][pred["scores"] > THRESH_V1].cpu()
labels_v1 = [CLASS_MAP[i] for i in pred["labels"][pred["scores"] > THRESH_V1].cpu()]
img_v1 = draw_bounding_boxes((img_t * 255).byte(), boxes_v1, labels_v1, colors="red")

# Рисуем V2
boxes_v2 = pred["boxes"][pred["scores"] > THRESH_V2].cpu()
labels_v2 = [CLASS_MAP[i] for i in pred["labels"][pred["scores"] > THRESH_V2].cpu()]
img_v2 = draw_bounding_boxes((img_t * 255).byte(), boxes_v2, labels_v2, colors="blue")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(img_v1.permute(1,2,0))
ax[0].set_title(f"V1 (thresh={THRESH_V1})")
ax[0].axis('off')
ax[1].imshow(img_v2.permute(1,2,0))
ax[1].set_title(f"V2 (thresh={THRESH_V2})")
ax[1].axis('off')
plt.savefig("artifacts/figures/detection_examples.png")
plt.show()

# Сохранение CSV
df = pd.DataFrame(results)
df.to_csv("artifacts/runs.csv", index=False)
print("Saved artifacts/runs.csv")